In [1]:
!pip install google-cloud-aiplatform

import vertexai
from vertexai.preview.generative_models import GenerativeModel, Part
from string import Template
from google.genai import types                    # for GenerateContentConfig
from pydantic import BaseModel                    # for GeminiStructuredOutput schema
import openai
from openai import OpenAI
import json
import re
import os
import numpy as np

# Schema for Gemini structured output — one root error per item
class GeminiStructuredOutput(BaseModel):
    error_name: str


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


/opt/conda/envs/llama_factory_2/lib/python3.12/site-packages/google/cloud/aiplatform/models.py:52: FutureWarning: Support for google-cloud-storage < 3.0.0 will be removed in a future version of google-cloud-aiplatform. Please upgrade to google-cloud-storage >= 3.0.0.
  from google.cloud.aiplatform.utils import gcs_utils


In [ ]:
vertexai.init(project="", location="")

GEMINI_MODEL = "gemini-3-flash-preview"
gemini_model = GenerativeModel(GEMINI_MODEL)
print("Gemini (Vertex AI) initialised.")

os.environ[""] = ""
openai_client = OpenAI(api_key=os.environ.get(""))
OPENAI_MODEL = "gpt-5.2"
print("OpenAI client initialised.")

Gemini (Vertex AI) initialised.
OpenAI client initialised.


/opt/conda/envs/llama_factory_2/lib/python3.12/site-packages/vertexai/generative_models/_generative_models.py:433: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


In [17]:
from pathlib import Path
from vertexai.generative_models import Part

def upload_pdf(pdf_path: str):
    pdf_path = Path(pdf_path)
    print(f"Processing: {pdf_path.name}")

    print("Loading PDF for Gemini...")
    with open(pdf_path, "rb") as f:
        pdf_bytes = f.read()
    gemini_pdf_part = Part.from_data(data=pdf_bytes, mime_type="application/pdf")
    print("Gemini PDF ready.")

    print("Uploading to OpenAI...")
    with open(pdf_path, "rb") as f:
        openai_file = openai_client.files.create(file=f, purpose="user_data")
    print(f"OpenAI file ready: {openai_file.id}")

    return gemini_pdf_part, openai_file.id

pdf_path = "06 SCA Trouble Shooting Guides Rev11072025.pptx.pdf"
gemini_pdf_part, openai_file_id = upload_pdf(pdf_path)

Processing: 06 SCA Trouble Shooting Guides Rev11072025.pptx.pdf
Loading PDF for Gemini...
Gemini PDF ready.
Uploading to OpenAI...
OpenAI file ready: file-SnF7hthts718mfwSvuPiR7


In [18]:
ROOT_FAULTS_PROMPT = Template("""
You are analyzing a technical troubleshooting document.

Your task: extract every top-level failure name from this document.

--------------------------------
HOW TO IDENTIFY THE RIGHT TABLES
--------------------------------
Troubleshooting documents typically have tables with 3 columns:
  Column 1: the failure or fault (what went wrong)
  Column 2: the cause or reason (why it happened)
  Column 3: the fix or treatment (what to do)

For EVERY table in the document:
  1. Look at the column structure.
  2. Identify which column describes 'what went wrong' — this is your target column.
     It could be called anything: Fault, Failure, Problem, Error, Issue, Symptom, etc.
  3. If a column contains short single-letter or single-number identifiers (like U, Q, E, 1, 2)
     as row values — that is an alarm/icon code column. SKIP that entire table.
  4. Otherwise extract every unique value from the 'what went wrong' column.

Do NOT extract causes, reasons, fixes, or corrective actions.

--------------------------------
Required Output Format
--------------------------------
Return ONLY valid JSON, no markdown, no explanation:

{
  "errors": [
    {"error_name": "<fault name>"},
    {"error_name": "<fault name>"}
  ]
}
""")

In [19]:
def extract_error_names_gemini(pdf_part):

    prompt = ROOT_FAULTS_PROMPT.substitute()

    # vertexai GenerativeModel requires a plain dict for generation_config
    response = gemini_model.generate_content(
        [pdf_part, prompt],
        generation_config={"temperature": 0, "top_p": 1.0, "top_k": 32,
                           "candidate_count": 1, "max_output_tokens": 65536}
    )

    text = response.text
    try:
        parsed = json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", text, re.DOTALL)
        parsed = json.loads(match.group()) if match else {}
    return parsed.get("errors", parsed) if isinstance(parsed, dict) else parsed

In [20]:
# ── STEP 4: Run Prompt 1 — Extract all root error names via Gemini ────────────
error_names = extract_error_names_gemini(gemini_pdf_part)

print(f"Extracted {len(error_names)} root errors:\n")
for e in error_names:
    print("-", e["error_name"])

Extracted 58 root errors:

- Lower limit value at program #
- Upper limit value at program #
- No material flow at program #
- No limit values at program #
- Bead interruption at program #
- PRESSURE UNDERRUN AT PROGRAM {#}
- PRESSURE OVERRUN AT PROGRAM {#}
- Invalid program parameters in program #
- Fill meter timeout
- Bead volume too small # in program # cm³
- Excessive bead volume # in program # cm³
- Meter # drained
- Servo drive error meter #
- Purge Process Interrupted
- Device error at node #
- System bleeding fault
- Pot Life Time Exceeded
- Stroke error: Meter #
- Meter # stroke deviation (mm)
- ON/OFF Response time of Applicator Valve is too long
- ON/OFF Response time of Fill Valve is too long
- Drive-Error: Manual mode meter #
- Drive error: Pre-pressurization meter #
- Drive-Error: Application (Flow) meter #
- Drive-Error: Application (Pressure) meter #
- Drive-Error: Filling meter #
- Fill valve defective or pump pressure too low meter {#}
- Upper end switch meter # does

In [21]:
# ── STEP 4b: Semantic Dedup — remove errors that mean the same thing ──────────
# Uses OpenAI embeddings + cosine similarity.
# Two errors with similarity >= THRESHOLD are considered duplicates; we keep the first.

SIMILARITY_THRESHOLD = 0.92  # tune this: higher = stricter dedup
EMBEDDING_MODEL = "text-embedding-3-small"

def get_embeddings(texts: list[str]) -> list[list[float]]:
    response = openai_client.embeddings.create(model=EMBEDDING_MODEL, input=texts)
    return [r.embedding for r in response.data]

def cosine_similarity(a, b) -> float:
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

def semantic_dedup(error_list: list, threshold: float = SIMILARITY_THRESHOLD) -> list:
    names = [e.get("error_name", str(e)) for e in error_list]
    embeddings = get_embeddings(names)

    kept = []   # indices we keep
    for i, emb_i in enumerate(embeddings):
        # drop if too similar to any already-kept error
        if any(cosine_similarity(emb_i, embeddings[j]) >= threshold for j in kept):
            print(f"  Dropped (duplicate): {names[i]}")
            continue
        kept.append(i)

    deduped = [error_list[i] for i in kept]
    print(f"\nDedup: {len(error_list)} → {len(deduped)} unique errors")
    return deduped

deduped_errors = semantic_dedup(error_names)

print("\nUnique Root Errors after dedup:")
for e in deduped_errors:
    print("-", e.get("error_name", e))

  Dropped (duplicate): Lower end switch meter # does not work

Dedup: 58 → 57 unique errors

Unique Root Errors after dedup:
- Lower limit value at program #
- Upper limit value at program #
- No material flow at program #
- No limit values at program #
- Bead interruption at program #
- PRESSURE UNDERRUN AT PROGRAM {#}
- PRESSURE OVERRUN AT PROGRAM {#}
- Invalid program parameters in program #
- Fill meter timeout
- Bead volume too small # in program # cm³
- Excessive bead volume # in program # cm³
- Meter # drained
- Servo drive error meter #
- Purge Process Interrupted
- Device error at node #
- System bleeding fault
- Pot Life Time Exceeded
- Stroke error: Meter #
- Meter # stroke deviation (mm)
- ON/OFF Response time of Applicator Valve is too long
- ON/OFF Response time of Fill Valve is too long
- Drive-Error: Manual mode meter #
- Drive error: Pre-pressurization meter #
- Drive-Error: Application (Flow) meter #
- Drive-Error: Application (Pressure) meter #
- Drive-Error: Filling

In [22]:
# ── STEP 5: Prompt 2 — Build a full fault tree for a single root error (GPT-5.2) ─
# Strictly grounded in the PDF — model is NOT allowed to use its own knowledge.

FAULT_TREE_PROMPT = Template("""
You are a fault tree analyst. You are given a technical troubleshooting document.

The following are ALL known top-level errors extracted from this document:
$all_errors

Your task is to build a fault tree ONLY for this TOP EVENT: "$error_name"
Do NOT include causes that belong to other top-level errors listed above.

Rules:
- Use ONLY information found in the provided document. Do NOT use your own knowledge.
- TOP EVENT = the root error (already given).
- CHILDREN = direct causes or contributing conditions listed in the document.
- LEAF NODES = the most specific causes with no further breakdown in the document.
- Each child that has sub-causes should also have its own children array.

Required Output Format — return ONLY valid JSON, no markdown, no explanation:

{
  "error_name": "$error_name",
  "children": [
    {
      "error_name": "<cause>",
      "children": [
        {"error_name": "<sub-cause>", "children": []}
      ]
    }
  ]
}
""")

def build_fault_tree_openai(error_name: str, file_id: str, all_error_names: list) -> dict:
    """Uses GPT-5.2 + the uploaded PDF to build a fault tree for one root error."""
    # Format the full error list as a numbered reference for the prompt
    all_errors_str = "\n".join(f"  {i+1}. {e['error_name']}" for i, e in enumerate(all_error_names))

    prompt = FAULT_TREE_PROMPT.substitute(error_name=error_name, all_errors=all_errors_str)

    response = openai_client.chat.completions.create(
        model=OPENAI_MODEL,
        temperature=0,
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "file", "file": {"file_id": file_id}},
                    {"type": "text",  "text": prompt}
                ]
            }
        ]
    )

    text = response.choices[0].message.content

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", text, re.DOTALL)
        return json.loads(match.group()) if match else {"error_name": error_name, "children": []}

In [ ]:
# ── STEP 6: Build fault trees for ALL root errors ─────────────────────────────

def build_all_fault_trees(error_list: list, file_id: str) -> dict:
    """Returns a dict mapping error_name → fault tree dict."""
    fault_trees = {}
    for i, e in enumerate(error_list):
        name = e.get("error_name", str(e))
        print(f"[{i+1}/{len(error_list)}] Building tree for: {name}")
        fault_trees[name] = build_fault_tree_openai(name, file_id, error_list)  # pass full list
    return fault_trees

all_fault_trees = build_all_fault_trees(deduped_errors, openai_file_id)

# ── Pretty ASCII tree printer ─────────────────────────────────────────────────
def print_tree(node: dict, prefix: str = "", is_last: bool = True):
    connector = "└── " if is_last else "├── "
    print(prefix + connector + node.get("error_name", "?"))
    children = node.get("children", [])
    for i, child in enumerate(children):
        extension = "    " if is_last else "│   "
        print_tree(child, prefix + extension, i == len(children) - 1)

# ── Print ALL fault trees — ASCII tree + JSON ─────────────────────────────────
print("\n=== ALL FAULT TREES ===\n")
for name, tree in all_fault_trees.items():
    print(f"\n{'='*60}")
    print(f"TOP EVENT: {name}")
    print('='*60)
    print("\n[Tree View]")
    print_tree(tree)
    print("\n[JSON View]")
    print(json.dumps(tree, indent=2))

# ── Save all fault trees to JSON file ────────────────────────────────────────
output_path = "fault_trees.json"
with open(output_path, "w") as f:
    json.dump(list(all_fault_trees.values()), f, indent=2)
print(f"\nSaved {len(all_fault_trees)} fault trees to {output_path}")

[1/57] Building tree for: Lower limit value at program #
[2/57] Building tree for: Upper limit value at program #
[3/57] Building tree for: No material flow at program #
